In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## Ejercicio 1

- $n$ muestras $\mathbf x_i\in\mathbb R^p$ con etiquetas $y_i\in\{0, 1,\dots,k-1\}$.
- $n_y=\sum_{i=1}^n\mathbf 1\{y_i=y\}$ es la cantidad de muestras de la clase $y$.
- Modelo generativo: $\mathbf X\mid Y=y\sim\mathcal N(\boldsymbol\mu_y,\Sigma_y)$ y $\mathbb P\{Y=y\}=\pi_y$.
- LDA supone $\Sigma_y=\Sigma$ para todas las clases (homocedasticidad), mientras que QDA permite una $\Sigma_y$ distinta por clase.

**Estimadores de máxima verosimilitud**

$$
\hat\pi_y=\frac{n_y}{n},\qquad
\hat{\boldsymbol\mu}_y=\frac{1}{n_y}\sum_{i:\,y_i=y}\mathbf x_i .
$$

En LDA la covarianza es compartida y se promedia sobre las $n$ muestras, cada una centrada en la media de su clase:

$$
\hat\Sigma=\frac1n\sum_{i=1}^n(\mathbf x_i-\hat{\boldsymbol\mu}_{y_i})(\mathbf x_i-\hat{\boldsymbol\mu}_{y_i})^\top .
$$

En QDA hay una covarianza por clase y el divisor es $n_y$:

$$
\hat\Sigma_y=\frac1{n_y}\sum_{i:\,y_i=y}(\mathbf x_i-\hat{\boldsymbol\mu}_{y})(\mathbf x_i-\hat{\boldsymbol\mu}_{y})^\top .
$$

**Regularización**

Si $\hat\Sigma$ es singular o está mal condicionada (por ejemplo, si $n_y<p$ o hay variables constantes), no la podemos invertir. 

Para eso la reemplazamos por $\hat\Sigma+\varepsilon I$ con $\varepsilon\ge0$, que sí es definida positiva para cualquier $\varepsilon>0$.

**Funciones discriminantes**

La regla de Bayes asigna $G(\mathbf x)=\arg\max_y\big[\log p(\mathbf x\mid y)+\log\pi_y\big]$. Con la densidad normal:

$$
\log p(\mathbf x\mid y)+\log\pi_y=
\underbrace{-\tfrac p2\log(2\pi)}_{\text{igual para todas}}
-\tfrac12\log|\Sigma_y|-\tfrac12(\mathbf x-\boldsymbol\mu_y)^\top\Sigma_y^{-1}(\mathbf x-\boldsymbol\mu_y)+\log\pi_y .
$$

Descartando lo que no depende de $y$, nos quedan las **funciones discriminantes** $\delta_y(\mathbf x)$:

- **QDA:**
$$
\delta_y(\mathbf x)=-\tfrac12\log|\Sigma_y|-\tfrac12(\mathbf x-\boldsymbol\mu_y)^\top\Sigma_y^{-1}(\mathbf x-\boldsymbol\mu_y)+\log\pi_y .
$$

- **LDA:** es la misma expresión con $\Sigma_y=\Sigma$. Como $\log|\Sigma|$ y el término $-\tfrac12\mathbf x^\top\Sigma^{-1}\mathbf x$ (que aparece al expandir la forma cuadrática) son comunes a todas las clases, al comparar clases solo importa la parte lineal en $\mathbf x$:
$$
\delta_y(\mathbf x)=\mathbf x^\top\Sigma^{-1}\boldsymbol\mu_y-\tfrac12\boldsymbol\mu_y^\top\Sigma^{-1}\boldsymbol\mu_y+\log\pi_y .
$$

Así clasificamos con $G(\mathbf x)=\arg\max_y\delta_y(\mathbf x)$ sin evaluar la densidad normal. En la implementación usamos la expresión de QDA para los tres modelos: con $\Sigma$ compartida da las mismas clases y probabilidades que la forma lineal, porque los términos que difieren son iguales en todas las clases.

- **Naive Bayes gaussiano** (ejercicio 3): es QDA con cada $\Sigma_y$ **diagonal**, $\hat\Sigma_y=\operatorname{diag}(\hat\sigma^2_{1y},\dots,\hat\sigma^2_{py})$, que equivale a suponer las variables independientes dentro de cada clase.

**Probabilidades a posteriori**

Como $\log p(\mathbf x\mid y)+\log\pi_y=\delta_y(\mathbf x)+c(\mathbf x)$, donde $c(\mathbf x)$ no depende de $y$, la constante se cancela al normalizar:

$$
p(y\mid\mathbf x)=\frac{e^{\delta_y(\mathbf x)}}{\sum_{l=1}^k e^{\delta_l(\mathbf x)}}\qquad\text{(softmax de los }\delta_y\text{)}.
$$

Para evitar desbordamientos en la exponencial, a todos los $\delta_l(\mathbf x)$ les restamos su máximo $\delta_{\max}(\mathbf x)$ antes de exponenciar (truco *log-sum-exp*). 

El cociente no cambia, porque $e^{-\delta_{\max}}$ aparece en el numerador y en el denominador.

**Organización del código.** Los tres modelos solo difieren en el supuesto sobre la covarianza, así que comparten casi todo:

- `myClasificadorGenerativo`: a partir de `decision_function` ($\delta_y(\mathbf x)$, de forma `(n, k)`), calcula `predict_proba` (softmax con log-sum-exp) y `predict`. La reutiliza también naive Bayes Bernoulli.
- `myDiscriminanteGaussiano(covarianza, reg_covar)`: estima $\hat\pi_y$, $\hat{\boldsymbol\mu}_y$ y las covarianzas según el supuesto (`"compartida"`, `"por_clase"` o `"diagonal"`), y calcula $\delta_y(\mathbf x)$.
- `myLDA`, `myQDA` y `myNaiveBayesGaussiano` (ejercicio 3): subclases que solo fijan el supuesto.

`reg_covar` es el $\varepsilon$ de la consigna: se suma $\varepsilon I$ a cada covarianza estimada (el mismo nombre que en `myGMM`, guía 3.3).

In [ ]:
class myClasificadorGenerativo:
    """
    Base de los clasificadores generativos: probabilidades y predicción a partir de
    δ_y(x) = log π_y + log p(x | y) (salvo una constante que no depende de y).
    """
    def decision_function(self, X):
        """
        Funciones discriminantes; la define cada subclase.

        Parámetros:
        - X: (n, p)

        Retorna:
        - delta: (n, k)
        """
        raise NotImplementedError

    def predict_proba(self, X):
        """
        Probabilidades a posteriori p(y | x): softmax de δ_y(x) con log-sum-exp.

        Parámetros:
        - X: (n, p)

        Retorna:
        - P: (n, k)
        """

        # TODO: restar a cada fila su máximo, exponenciar y normalizar

        delta = ...  # (n, k)
        delta = ...  # (n, k)
        P = ...  # (n, k)
        return ...

    def predict(self, X):
        """
        Clase de mayor δ_y(x).

        Parámetros:
        - X: (n, p)

        Retorna:
        - y_pred: (n,)
        """

        # TODO

        return ...  # (n,)


class myDiscriminanteGaussiano(myClasificadorGenerativo):
    """
    Clasificador con X | Y = y ~ N(μ_y, Σ_y), con un supuesto sobre las covarianzas.
    """
    def __init__(self, covarianza="por_clase", reg_covar=0.0):
        """
        Inicializa el modelo.

        Parámetros:
        - covarianza: "compartida" (LDA), "por_clase" (QDA) o "diagonal" (naive Bayes gaussiano)
        - reg_covar: ε >= 0 que se suma a la diagonal de cada covarianza
        """

        if covarianza not in ("compartida", "por_clase", "diagonal"):
            raise ValueError(f"covarianza no válida: {covarianza!r}")
        self.covarianza = covarianza
        self.reg_covar = reg_covar

        # Resultados del ajuste
        self.classes = None  # etiquetas de las clases (k,)
        self.pi = None  # probabilidades a priori (k,)
        self.mu = None  # medias (k, p)
        self.Sigma = None  # covarianzas (k, p, p); en LDA, la misma para todas
        self.Sigma_inv = None  # inversas (k, p, p)
        self.logdet = None  # log|Σ_y| (k,)

    def fit(self, X, y):
        """
        Estima los parámetros por máxima verosimilitud.

        Parámetros:
        - X: (n, p)
        - y: (n,)

        Retorna:
        - self
        """

        # TODO:
        # - π̂_y y μ̂_y de cada clase
        # - Covarianzas según el supuesto (divisor n en la compartida, n_y en las de cada clase)
        # - Sumar ε I, invertir y calcular log|Σ_y| (con np.linalg.slogdet)

        n, p = X.shape
        self.classes = np.unique(y)
        k = len(self.classes)
        indices = np.searchsorted(self.classes, y)  # clase de cada muestra como 0, ..., k-1, (n,)

        self.pi = np.zeros(k)
        self.mu = np.zeros((k, p))
        for j in range(k):
            Xj = X[indices == j]  # (n_j, p)
            self.pi[j] = ...
            self.mu[j] = ...  # (p,)

        Z = ...  # cada muestra centrada en la media de su clase, (n, p)
        if self.covarianza == "compartida":
            # Opción 1: muestra a muestra
            S = np.zeros((p, p))
            for i in range(n):
                S += ...  # (p, p)
            S /= n

            # Opción 2: vectorizada, (p, n) @ (n, p) -> (p, p)
            S = ...
            Sigma = np.array([S] * k)  # (k, p, p)
        else:
            Sigma = np.zeros((k, p, p))
            for j in range(k):
                Zj = Z[indices == j]  # (n_j, p)
                Sigma[j] = ...  # (p, p)
                if self.covarianza == "diagonal":
                    Sigma[j] = ...  # solo la diagonal

        self.Sigma = Sigma + self.reg_covar * np.eye(p)  # (k, p, p)
        self.Sigma_inv = ...  # (k, p, p)
        self.logdet = ...  # (k,)
        return self

    def decision_function(self, X):
        """
        δ_y(x) = -½ log|Σ_y| - ½ (x - μ_y)^T Σ_y^{-1} (x - μ_y) + log π_y.

        Parámetros:
        - X: (n, p)

        Retorna:
        - delta: (n, k)
        """

        # TODO: distancia de Mahalanobis de cada muestra a cada media

        delta = np.zeros((X.shape[0], len(self.classes)))  # (n, k)
        for j in range(len(self.classes)):
            D = ...  # (n, p)

            # Opción 1: muestra a muestra
            m = np.zeros(X.shape[0])  # (n,)
            for i in range(X.shape[0]):
                m[i] = ...

            # Opción 2: vectorizada, ((n, p) @ (p, p)) * (n, p) -> suma por filas -> (n,)
            m = ...

            delta[:, j] = ...
        return delta


class myLDA(myDiscriminanteGaussiano):
    """
    Análisis discriminante lineal: covarianza compartida.
    """
    def __init__(self, reg_covar=0.0):
        super().__init__(covarianza="compartida", reg_covar=reg_covar)


class myQDA(myDiscriminanteGaussiano):
    """
    Análisis discriminante cuadrático: una covarianza por clase.
    """
    def __init__(self, reg_covar=0.0):
        super().__init__(covarianza="por_clase", reg_covar=reg_covar)

##### Solución

In [ ]:
class myClasificadorGenerativo:
    """
    Base de los clasificadores generativos: probabilidades y predicción a partir de
    δ_y(x) = log π_y + log p(x | y) (salvo una constante que no depende de y).
    """
    def decision_function(self, X):
        """
        Funciones discriminantes; la define cada subclase.

        Parámetros:
        - X: (n, p)

        Retorna:
        - delta: (n, k)
        """
        raise NotImplementedError

    def predict_proba(self, X):
        """
        Probabilidades a posteriori p(y | x): softmax de δ_y(x) con log-sum-exp.

        Parámetros:
        - X: (n, p)

        Retorna:
        - P: (n, k)
        """

        delta = self.decision_function(X)  # (n, k)
        delta = delta - np.max(delta, axis=1, keepdims=True)  # (n, k)
        P = np.exp(delta)  # (n, k)
        return P / np.sum(P, axis=1, keepdims=True)

    def predict(self, X):
        """
        Clase de mayor δ_y(x).

        Parámetros:
        - X: (n, p)

        Retorna:
        - y_pred: (n,)
        """

        return self.classes[np.argmax(self.decision_function(X), axis=1)]  # (n,)


class myDiscriminanteGaussiano(myClasificadorGenerativo):
    """
    Clasificador con X | Y = y ~ N(μ_y, Σ_y), con un supuesto sobre las covarianzas.
    """
    def __init__(self, covarianza="por_clase", reg_covar=0.0):
        """
        Inicializa el modelo.

        Parámetros:
        - covarianza: "compartida" (LDA), "por_clase" (QDA) o "diagonal" (naive Bayes gaussiano)
        - reg_covar: ε >= 0 que se suma a la diagonal de cada covarianza
        """

        if covarianza not in ("compartida", "por_clase", "diagonal"):
            raise ValueError(f"covarianza no válida: {covarianza!r}")
        self.covarianza = covarianza
        self.reg_covar = reg_covar

        # Resultados del ajuste
        self.classes = None  # etiquetas de las clases (k,)
        self.pi = None  # probabilidades a priori (k,)
        self.mu = None  # medias (k, p)
        self.Sigma = None  # covarianzas (k, p, p); en LDA, la misma para todas
        self.Sigma_inv = None  # inversas (k, p, p)
        self.logdet = None  # log|Σ_y| (k,)

    def fit(self, X, y):
        """
        Estima los parámetros por máxima verosimilitud.

        Parámetros:
        - X: (n, p)
        - y: (n,)

        Retorna:
        - self
        """

        n, p = X.shape
        self.classes = np.unique(y)
        k = len(self.classes)
        indices = np.searchsorted(self.classes, y)  # clase de cada muestra como 0, ..., k-1, (n,)

        self.pi = np.zeros(k)
        self.mu = np.zeros((k, p))
        for j in range(k):
            Xj = X[indices == j]  # (n_j, p)
            self.pi[j] = Xj.shape[0] / n
            self.mu[j] = np.mean(Xj, axis=0)  # (p,)

        Z = X - self.mu[indices]  # cada muestra centrada en la media de su clase, (n, p)
        if self.covarianza == "compartida":
            # Opción 1: muestra a muestra
            S = np.zeros((p, p))
            for i in range(n):
                S += np.outer(Z[i], Z[i])  # (p, p)
            S /= n

            # Opción 2: vectorizada, (p, n) @ (n, p) -> (p, p)
            S = Z.T @ Z / n
            Sigma = np.array([S] * k)  # (k, p, p)
        else:
            Sigma = np.zeros((k, p, p))
            for j in range(k):
                Zj = Z[indices == j]  # (n_j, p)
                Sigma[j] = Zj.T @ Zj / Zj.shape[0]  # (p, p)
                if self.covarianza == "diagonal":
                    Sigma[j] = np.diag(np.diag(Sigma[j]))  # solo la diagonal

        self.Sigma = Sigma + self.reg_covar * np.eye(p)  # (k, p, p)
        self.Sigma_inv = np.linalg.inv(self.Sigma)  # (k, p, p)
        self.logdet = np.linalg.slogdet(self.Sigma)[1]  # (k,)
        return self

    def decision_function(self, X):
        """
        δ_y(x) = -½ log|Σ_y| - ½ (x - μ_y)^T Σ_y^{-1} (x - μ_y) + log π_y.

        Parámetros:
        - X: (n, p)

        Retorna:
        - delta: (n, k)
        """

        delta = np.zeros((X.shape[0], len(self.classes)))  # (n, k)
        for j in range(len(self.classes)):
            D = X - self.mu[j]  # (n, p)

            # Opción 1: muestra a muestra
            m = np.zeros(X.shape[0])  # (n,)
            for i in range(X.shape[0]):
                m[i] = D[i] @ self.Sigma_inv[j] @ D[i]

            # Opción 2: vectorizada, ((n, p) @ (p, p)) * (n, p) -> suma por filas -> (n,)
            m = np.sum((D @ self.Sigma_inv[j]) * D, axis=1)

            delta[:, j] = -0.5 * self.logdet[j] - 0.5 * m + np.log(self.pi[j])
        return delta


class myLDA(myDiscriminanteGaussiano):
    """
    Análisis discriminante lineal: covarianza compartida.
    """
    def __init__(self, reg_covar=0.0):
        super().__init__(covarianza="compartida", reg_covar=reg_covar)


class myQDA(myDiscriminanteGaussiano):
    """
    Análisis discriminante cuadrático: una covarianza por clase.
    """
    def __init__(self, reg_covar=0.0):
        super().__init__(covarianza="por_clase", reg_covar=reg_covar)

## Ejercicio 2

Las fronteras de decisión son los puntos donde empatan las dos funciones discriminantes más grandes, $\delta_y(\mathbf x)=\delta_l(\mathbf x)$:

- En **LDA**, $\delta_y-\delta_l$ es afín en $\mathbf x$. Las fronteras son rectas (hiperplanos).
- En **QDA**, $\delta_y-\delta_l$ conserva el término $-\tfrac12\mathbf x^\top(\Sigma_y^{-1}-\Sigma_l^{-1})\mathbf x$. Las fronteras son cónicas: elipses, parábolas o hipérbolas.

Para graficarlas en $\mathbb R^2$, evaluamos `predict` sobre una grilla y pintamos las regiones de cada clase. Las fronteras quedan en los bordes entre regiones.

En todos los escenarios generamos $n$ muestras por clase, separamos 30 % para test y reportamos el accuracy en ambos conjuntos.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

def generar_normales(mus, Sigmas, n_por_clase=200, seed=0):
    """
    Genera n_por_clase muestras de N(mu_y, Sigma_y) para cada clase y = 0, ..., k-1.

    Retorna:
    - X: datos (k * n_por_clase, p)
    - y: etiquetas (k * n_por_clase,)
    """
    rng = np.random.default_rng(seed)
    X = np.vstack([rng.multivariate_normal(mu, S, size=n_por_clase) for mu, S in zip(mus, Sigmas)])
    y = np.repeat(np.arange(len(mus)), n_por_clase)
    return X, y

In [ ]:
def plot_fronteras(model, X, y, ax, title=""):
    """
    Grafica las regiones de decisión de un modelo ya ajustado y los datos.
    """

    # Grilla sobre el plano
    x1 = np.linspace(X[:, 0].min() - 1, X[:, 0].max() + 1, 300)
    x2 = np.linspace(X[:, 1].min() - 1, X[:, 1].max() + 1, 300)
    xx1, xx2 = np.meshgrid(x1, x2)

    # Clase predicha en cada punto de la grilla (como índice 0, ..., k-1)
    Z = model.predict(np.c_[xx1.ravel(), xx2.ravel()])
    Z = np.searchsorted(model.classes, Z).reshape(xx1.shape)

    k = len(model.classes)
    colores = ["tab:blue", "tab:red", "tab:green", "tab:orange"][:k]

    # Regiones de decisión y fronteras (bordes entre regiones)
    ax.contourf(xx1, xx2, Z, levels=np.arange(k + 1) - 0.5, colors=colores, alpha=0.15)
    ax.contour(xx1, xx2, Z, levels=np.arange(k) + 0.5, colors="k", linewidths=1)

    # Datos
    for j, c in enumerate(model.classes):
        ax.scatter(X[y == c, 0], X[y == c, 1], c=colores[j], s=10, label=f"Clase {c}")

    ax.set_title(title)
    ax.set_xlabel("x1")
    ax.set_ylabel("x2")


def comparar_lda_qda(X, y, titulo):
    """
    Ajusta myLDA y myQDA, reporta el accuracy y grafica ambas fronteras.
    """

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y)

    modelos = {
        "LDA": myLDA().fit(X_train, y_train),
        "QDA": myQDA().fit(X_train, y_train),
    }

    fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)

    for ax, (nombre, model) in zip(axes, modelos.items()):
        acc_train = accuracy_score(y_train, model.predict(X_train))
        acc_test = accuracy_score(y_test, model.predict(X_test))
        print(f"{titulo} | {nombre} | Accuracy Train: {acc_train:.4f} | Accuracy Test: {acc_test:.4f}")

        plot_fronteras(
            model,
            X_train,
            y_train,
            ax,
            title=f"{nombre} | acc test = {acc_test:.3f}",
        )

    axes[0].legend(loc="best", fontsize=8)
    fig.suptitle(titulo)
    plt.tight_layout()
    plt.show()

    return modelos, (X_train, X_test, y_train, y_test)

#### a) Dos clases esféricas

In [ ]:
I = np.eye(2)

mus_a = [np.array([-2.0, 0.0]), np.array([2.0, 0.0])]
Sigmas_a = [I, I]

X_a, y_a = generar_normales(mus_a, Sigmas_a, n_por_clase=200, seed=0)
modelos_a, datos_a = comparar_lda_qda(X_a, y_a, "a) Dos clases esféricas")

#### b) Tres clases esféricas

In [ ]:
mus_b = [np.array([0.0, 3.0]), np.array([-3.0, -2.0]), np.array([3.0, -2.0])]
Sigmas_b = [I, I, I]

X_b, y_b = generar_normales(mus_b, Sigmas_b, n_por_clase=200, seed=1)
modelos_b, datos_b = comparar_lda_qda(X_b, y_b, "b) Tres clases esféricas")

#### c) Tres clases con covarianzas distintas

In [ ]:
mus_c = [np.array([0.0, 0.0]), np.array([0.0, 3.0]), np.array([0.0, -3.0])]
Sigmas_c = [
    np.array([[1.0, 0.0], [0.0, 1.0]]),
    np.array([[5.0, -4.5], [-4.5, 5.0]]),
    np.array([[5.0, 4.5], [4.5, 5.0]]),
]

X_c, y_c = generar_normales(mus_c, Sigmas_c, n_por_clase=200, seed=2)
modelos_c, datos_c = comparar_lda_qda(X_c, y_c, "c) Covarianzas distintas")

#### d) *(Adicional)* Clases no separables linealmente

In [ ]:
from sklearn.datasets import make_moons

X_d, y_d = make_moons(n_samples=400, noise=0.2, random_state=0)
modelos_d, datos_d = comparar_lda_qda(X_d, y_d, "d) make_moons")

#### e) ¿Cuándo alcanza LDA, cuándo conviene QDA y cuándo fallan los dos?

#### f) Comparación con scikit-learn

En `scikit-learn` 1.9, `LinearDiscriminantAnalysis(store_covariance=True)` guarda en `covariance_` la covarianza *within-class* con divisor $n$, y `QuadraticDiscriminantAnalysis(store_covariance=True)` usa $\hat\Sigma_y$ con divisor $n_y$: los mismos estimadores de máxima verosimilitud que usamos nosotros. (Versiones anteriores dividían por $n_y-1$ en QDA, y las probabilidades diferían levemente.)

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis

escenarios = {"a": (modelos_a, datos_a), "b": (modelos_b, datos_b), "c": (modelos_c, datos_c)}

for nombre, (modelos, (X_train, X_test, y_train, y_test)) in escenarios.items():

    lda, qda = modelos["LDA"], modelos["QDA"]

    lda_sk = LinearDiscriminantAnalysis(store_covariance=True).fit(X_train, y_train)
    qda_sk = QuadraticDiscriminantAnalysis(store_covariance=True).fit(X_train, y_train)

    Sigma_qda_sk = np.array(qda_sk.covariance_)

    print(f"Escenario {nombre})")
    print(f"  máx. |π̂ - π̂_sk| = {np.max(np.abs(lda.pi - lda_sk.priors_)):.2e}")
    print(f"  máx. |μ̂ - μ̂_sk| = {np.max(np.abs(lda.mu - lda_sk.means_)):.2e}")
    print(f"  máx. |Σ̂ - Σ̂_sk| (LDA) = {np.max(np.abs(lda.Sigma[0] - lda_sk.covariance_)):.2e}")
    print(f"  máx. |Σ̂_y - Σ̂_y,sk| (QDA) = {np.max(np.abs(qda.Sigma - Sigma_qda_sk)):.2e}")

    for etiqueta, mine, sk in [("LDA", lda, lda_sk), ("QDA", qda, qda_sk)]:
        coinc = np.mean(mine.predict(X_test) == sk.predict(X_test))
        dif_P = np.max(np.abs(mine.predict_proba(X_test) - sk.predict_proba(X_test)))
        print(f"  {etiqueta}: coincidencia de predicciones en test = {coinc:.4f}, máx. |p - p_sk| = {dif_P:.2e}")
    print()

## Ejercicio 3

**Supuesto naive Bayes:** dada la clase, las variables son independientes:

$$
p(\mathbf x\mid y)=\prod_{j=1}^p p(x_j\mid y)
\quad\Longrightarrow\quad
\log p(\mathbf x\mid y)+\log\pi_y=\log\pi_y+\sum_{j=1}^p\log p(x_j\mid y).
$$

Trabajamos con la **suma de logaritmos** y no con el producto. Con $p$ grande (miles de palabras en texto), el producto de $p$ probabilidades menores que 1 da `0.0` en punto flotante (*underflow*). Después, las probabilidades a posteriori se obtienen igual que en el Ejercicio 1: softmax de $\delta_y(\mathbf x)=\log\pi_y+\log p(\mathbf x\mid y)$, restando el máximo de cada fila.

**Variante gaussiana:** $X_j\mid Y=y\sim\mathcal N(\mu_{jy},\sigma^2_{jy})$, con estimadores MV

$$
\hat\mu_{jy}=\frac1{n_y}\sum_{i:\,y_i=y}x_{ij},\qquad
\hat\sigma^2_{jy}=\frac1{n_y}\sum_{i:\,y_i=y}(x_{ij}-\hat\mu_{jy})^2,
$$

$$
\log p(x_j\mid y)=-\tfrac12\log(2\pi\hat\sigma^2_{jy})-\frac{(x_j-\hat\mu_{jy})^2}{2\hat\sigma^2_{jy}} .
$$

Si una variable es constante dentro de una clase, entonces $\hat\sigma^2_{jy}=0$ y la expresión no está definida. Como en `GaussianNB` de scikit-learn, sumamos a todas las varianzas una cantidad pequeña: `var_smoothing` $\times\max_j\widehat{\mathrm{Var}}(X_j)$.

**Variante Bernoulli:** $X_j\mid Y=y\sim\mathrm{Bernoulli}(\phi_{j|y})$, con $x_j\in\{0,1\}$. El estimador con suavizado $\alpha\ge0$ es

$$
\hat\phi_{j|y}=\frac{\sum_{i=1}^n\mathbf 1\{x_{ij}=1\wedge y_i=y\}+\alpha}{n_y+2\alpha},
$$

- $\alpha=0$ es el MLE sin suavizar.
- $\alpha=1$ es el suavizado de Laplace (equivale a una prior $\mathrm{Beta}(1,1)$ sobre $\phi_{j|y}$).

$$
\log p(\mathbf x\mid y)=\sum_{j=1}^p\Big[x_j\log\hat\phi_{j|y}+(1-x_j)\log(1-\hat\phi_{j|y})\Big]
=\mathbf x^\top\log\hat{\boldsymbol\phi}_{y}+(\mathbf 1-\mathbf x)^\top\log(\mathbf 1-\hat{\boldsymbol\phi}_{y}).
$$

Con $\alpha=0$ puede pasar que $\hat\phi_{j|y}=0$ (o $=1$), y entonces $\log0=-\infty$. Además, en el producto matricial aparece $0\cdot(-\infty)=\texttt{nan}$. Para evitarlo reemplazamos $\log0$ por un número finito muy negativo (`LOG0 = -1e10`). El efecto es el mismo: esa clase queda con probabilidad a posteriori 0.

#### Naive Bayes gaussiano

Es `myDiscriminanteGaussiano` con covarianzas diagonales: alcanza con una subclase que fije el supuesto. (Con matrices diagonales, invertir y calcular el determinante es trivial, pero usamos el mismo código que en QDA para no duplicarlo; con $p$ muy grande convendría trabajar solo con las varianzas.)

In [ ]:
class myNaiveBayesGaussiano(myDiscriminanteGaussiano):
    """
    Naive Bayes gaussiano: QDA con covarianzas diagonales.
    """
    def __init__(self, reg_covar=0.0):
        """
        Inicializa el modelo.

        Parámetros:
        - reg_covar: ε >= 0 que se suma a cada varianza
        """

        # TODO: llamar al constructor de la clase base con el supuesto correspondiente

        ...

##### Solución

In [ ]:
class myNaiveBayesGaussiano(myDiscriminanteGaussiano):
    """
    Naive Bayes gaussiano: QDA con covarianzas diagonales.
    """
    def __init__(self, reg_covar=0.0):
        """
        Inicializa el modelo.

        Parámetros:
        - reg_covar: ε >= 0 que se suma a cada varianza
        """

        super().__init__(covarianza="diagonal", reg_covar=reg_covar)

#### *(Adicional)* Variante Bernoulli

Para variables binarias, $p(x_j\mid y)=\phi_{j|y}^{x_j}(1-\phi_{j|y})^{1-x_j}$ y

$$
\delta_y(\mathbf x)=\log\pi_y+\sum_{j=1}^p\big[x_j\log\phi_{j|y}+(1-x_j)\log(1-\phi_{j|y})\big],
\qquad
\hat\phi_{j|y}=\frac{\#\{i:\ x_{ij}=1,\ y_i=y\}+\alpha}{n_y+2\alpha}.
$$

Con $\alpha=0$ es el MLE; con $\alpha=1$, el suavizado de Laplace. Si $\hat\phi_{j|y}$ vale 0 o 1, algún logaritmo es $-\infty$: lo reemplazamos por un número negativo muy grande (`LOG0`) para que las cuentas sigan siendo finitas. La clase hereda `predict_proba` y `predict` de `myClasificadorGenerativo`.

In [ ]:
class myNaiveBayesBernoulli(myClasificadorGenerativo):
    """
    Naive Bayes para variables binarias, con suavizado.
    """
    LOG0 = -1e10  # reemplazo finito de log(0)

    def __init__(self, alpha=1.0):
        """
        Inicializa el modelo.

        Parámetros:
        - alpha: suavizado (0: MLE, 1: Laplace)
        """

        self.alpha = alpha

        # Resultados del ajuste
        self.classes = None  # etiquetas de las clases (k,)
        self.pi = None  # probabilidades a priori (k,)
        self.phi = None  # φ_{j|y} (k, p)

    def fit(self, X, y):
        """
        Estima π̂_y y φ̂_{j|y}.

        Parámetros:
        - X: (n, p), con valores 0 y 1
        - y: (n,)

        Retorna:
        - self
        """

        # TODO: π̂_y y φ̂_{j|y} con suavizado α

        n, p = X.shape
        self.classes = np.unique(y)
        k = len(self.classes)
        self.pi = np.zeros(k)
        self.phi = np.zeros((k, p))
        for j, c in enumerate(self.classes):
            Xc = X[y == c]  # (n_c, p)
            self.pi[j] = ...
            self.phi[j] = ...  # (p,)
        return self

    def decision_function(self, X):
        """
        δ_y(x) = log π_y + x^T log φ_y + (1 - x)^T log(1 - φ_y).

        Parámetros:
        - X: (n, p)

        Retorna:
        - delta: (n, k)
        """

        # TODO: logaritmos de φ y de 1 - φ, reemplazando -inf por LOG0

        with np.errstate(divide="ignore"):  # log(0) da -inf: lo reemplazamos abajo
            log_phi = ...  # (k, p)
            log_1mphi = ...  # (k, p)
        log_phi[np.isinf(log_phi)] = self.LOG0
        log_1mphi[np.isinf(log_1mphi)] = self.LOG0

        # (n, p) @ (p, k) -> (n, k)
        return ...

##### Solución

In [ ]:
class myNaiveBayesBernoulli(myClasificadorGenerativo):
    """
    Naive Bayes para variables binarias, con suavizado.
    """
    LOG0 = -1e10  # reemplazo finito de log(0)

    def __init__(self, alpha=1.0):
        """
        Inicializa el modelo.

        Parámetros:
        - alpha: suavizado (0: MLE, 1: Laplace)
        """

        self.alpha = alpha

        # Resultados del ajuste
        self.classes = None  # etiquetas de las clases (k,)
        self.pi = None  # probabilidades a priori (k,)
        self.phi = None  # φ_{j|y} (k, p)

    def fit(self, X, y):
        """
        Estima π̂_y y φ̂_{j|y}.

        Parámetros:
        - X: (n, p), con valores 0 y 1
        - y: (n,)

        Retorna:
        - self
        """

        n, p = X.shape
        self.classes = np.unique(y)
        k = len(self.classes)
        self.pi = np.zeros(k)
        self.phi = np.zeros((k, p))
        for j, c in enumerate(self.classes):
            Xc = X[y == c]  # (n_c, p)
            self.pi[j] = Xc.shape[0] / n
            self.phi[j] = (np.sum(Xc, axis=0) + self.alpha) / (Xc.shape[0] + 2 * self.alpha)  # (p,)
        return self

    def decision_function(self, X):
        """
        δ_y(x) = log π_y + x^T log φ_y + (1 - x)^T log(1 - φ_y).

        Parámetros:
        - X: (n, p)

        Retorna:
        - delta: (n, k)
        """

        with np.errstate(divide="ignore"):  # log(0) da -inf: lo reemplazamos abajo
            log_phi = np.log(self.phi)  # (k, p)
            log_1mphi = np.log(1 - self.phi)  # (k, p)
        log_phi[np.isinf(log_phi)] = self.LOG0
        log_1mphi[np.isinf(log_1mphi)] = self.LOG0

        # (n, p) @ (p, k) -> (n, k)
        return np.log(self.pi) + X @ log_phi.T + (1 - X) @ log_1mphi.T

## Ejercicio 4 *(Adicional)*

Usamos cuatro categorías bien distintas. Con `remove=("headers", "footers", "quotes")` sacamos los encabezados, firmas y citas de cada mensaje. Así el modelo no puede "hacer trampa" con el nombre del foro o del autor, y tiene que clasificar por el contenido del texto.

In [ ]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

categorias = ["alt.atheism", "comp.graphics", "rec.sport.hockey", "sci.space"]

news_train = fetch_20newsgroups(
    subset="train",
    categories=categorias,
    remove=("headers", "footers", "quotes"),
    random_state=42,
)
news_test = fetch_20newsgroups(
    subset="test",
    categories=categorias,
    remove=("headers", "footers", "quotes"),
    random_state=42,
)

y_train_txt, y_test_txt = news_train.target, news_test.target
print(f"Documentos train: {len(news_train.data)}, test: {len(news_test.data)}")
print(f"Clases: {news_train.target_names}")
print(f"Documentos por clase (train): {np.bincount(y_train_txt)}")

#### a) Vectorización binaria y ajuste

In [ ]:
# Vocabulario: las max_features palabras más frecuentes en train (sin stop words en inglés)
# binary=True: x_j = 1 si la palabra j aparece en el documento, 0 si no
vectorizer = CountVectorizer(binary=True, max_features=2000, stop_words="english")

X_train_txt = vectorizer.fit_transform(news_train.data).toarray()
X_test_txt = vectorizer.transform(news_test.data).toarray()
vocabulario = vectorizer.get_feature_names_out()

print(f"X_train: {X_train_txt.shape}, X_test: {X_test_txt.shape}")
print(f"Ejemplo de palabras del vocabulario: {vocabulario[::200]}")

In [ ]:
nb_txt = myNaiveBayesBernoulli(alpha=1.0)
nb_txt.fit(X_train_txt, y_train_txt)

y_pred_txt = nb_txt.predict(X_test_txt)
print(f"Accuracy train (α = 1): {accuracy_score(y_train_txt, nb_txt.predict(X_train_txt)):.4f}")
print(f"Accuracy test (α = 1): {accuracy_score(y_test_txt, y_pred_txt):.4f}")

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(
    confusion_matrix(y_test_txt, y_pred_txt),
    display_labels=news_train.target_names,
).plot(ax=ax, colorbar=False, xticks_rotation=30)
ax.set_title("myNaiveBayes Bernoulli (α = 1) - test")
plt.tight_layout()
plt.show()

Para interpretar el modelo, miramos las palabras más características de cada clase: las de mayor $\log\hat\phi_{j|y}-\log\hat\phi_{j|\text{resto}}$, es decir, las que son mucho más probables en esa clase que en el promedio de las otras.

In [ ]:
for j, nombre in enumerate(news_train.target_names):
    phi_resto = np.mean(np.delete(nb_txt.phi, j, axis=0), axis=0)
    score = np.log(nb_txt.phi[j]) - np.log(phi_resto)
    top = vocabulario[np.argsort(score)[::-1][:10]]
    print(f"{nombre}: {', '.join(top)}")

#### b) Comparación con `BernoulliNB`

In [ ]:
from sklearn.naive_bayes import BernoulliNB

nb_sk = BernoulliNB(alpha=1.0).fit(X_train_txt, y_train_txt)

print(f"Accuracy en test: myNaiveBayesBernoulli {accuracy_score(y_test_txt, nb_txt.predict(X_test_txt)):.4f}, BernoulliNB {accuracy_score(y_test_txt, nb_sk.predict(X_test_txt)):.4f}")
print(f"máx. |φ̂ - φ̂_sk| = {np.max(np.abs(nb_txt.phi - np.exp(nb_sk.feature_log_prob_))):.2e}")
print(f"máx. |π̂ - π̂_sk| = {np.max(np.abs(nb_txt.pi - np.exp(nb_sk.class_log_prior_))):.2e}")
print(f"Predicciones de test iguales: {np.mean(nb_txt.predict(X_test_txt) == nb_sk.predict(X_test_txt)):.4f}")
print(f"máx. |p(y | x) - p_sk(y | x)| = {np.max(np.abs(nb_txt.predict_proba(X_test_txt) - nb_sk.predict_proba(X_test_txt))):.2e}")

#### c) Efecto del suavizado: $\alpha=0$ vs $\alpha=1$

In [ ]:
nb_0 = myNaiveBayesBernoulli(alpha=0.0).fit(X_train_txt, y_train_txt)
nb_1 = myNaiveBayesBernoulli(alpha=1.0).fit(X_train_txt, y_train_txt)

for alpha, model in [(0, nb_0), (1, nb_1)]:
    acc_train = accuracy_score(y_train_txt, model.predict(X_train_txt))
    acc_test = accuracy_score(y_test_txt, model.predict(X_test_txt))
    print(f"α = {alpha} | Accuracy Train: {acc_train:.4f} | Accuracy Test: {acc_test:.4f}")

Buscamos documentos de test que contengan alguna palabra que **nunca** apareció en entrenamiento en documentos de su clase verdadera. Para esas palabras, con $\alpha=0$ vale $\hat\phi_{j|y}=0$.

In [ ]:
# Para cada documento de test: ¿cuántas palabras presentes tienen φ̂_{j|y_true} = 0 con α = 0?
phi0_clase_real = nb_0.phi[y_test_txt]                         # (n_test, p)
no_vistas = (X_test_txt == 1) & (phi0_clase_real == 0)         # (n_test, p)
n_no_vistas = no_vistas.sum(axis=1)

# Lo mismo pero para todas las clases a la vez: documentos "imposibles" para todas las clases
imposible_todas = np.all((X_test_txt @ (nb_0.phi == 0).T) > 0, axis=1)

print(
    f"Documentos de test con al menos una palabra no vista en su clase verdadera: "
    f"{np.sum(n_no_vistas > 0)} de {len(y_test_txt)}",
)
print(f"Documentos de test con alguna palabra no vista en TODAS las clases: {np.sum(imposible_todas)}")

In [ ]:
# Tomamos un documento que:
# - tenga una única palabra no vista en su clase verdadera,
# - no sea "imposible" para todas las clases,
# - esté bien clasificado con α = 1
cand = (n_no_vistas == 1) & ~imposible_todas & (nb_1.predict(X_test_txt) == y_test_txt)
i = np.where(cand)[0][0]
j_word = np.where(no_vistas[i])[0][0]
clase_real = y_test_txt[i]
nombres = news_train.target_names

print(f"Documento de test {i} | clase verdadera: {nombres[clase_real]}")
print(f"Palabra no vista en '{nombres[clase_real]}': '{vocabulario[j_word]}'")
print(f"Apariciones de '{vocabulario[j_word]}' en train por clase: {[int(X_train_txt[y_train_txt == c, j_word].sum()) for c in range(len(nombres))]}")
print()
print("Fragmento del documento:")
print(news_test.data[i][:600])

In [ ]:
with np.printoptions(precision=4, suppress=True):
    print(f"φ̂_(j|y) de '{vocabulario[j_word]}' con α = 0: {nb_0.phi[:, j_word]}")
    print(f"φ̂_(j|y) de '{vocabulario[j_word]}' con α = 1: {nb_1.phi[:, j_word]}")
    print()
    print(f"p(y | x) con α = 0: {nb_0.predict_proba(X_test_txt[i:i+1])[0]}  -> predice: {nombres[nb_0.predict(X_test_txt[i:i+1])[0]]}")
    print(f"p(y | x) con α = 1: {nb_1.predict_proba(X_test_txt[i:i+1])[0]}  -> predice: {nombres[nb_1.predict(X_test_txt[i:i+1])[0]]}")

In [ ]:
# Consecuencias globales de α = 0 en test
P0 = nb_0.predict_proba(X_test_txt)
P1 = nb_1.predict_proba(X_test_txt)
p_real_0 = P0[np.arange(len(y_test_txt)), y_test_txt]

print(f"Documentos con p(y_true | x) = 0 exacto (α = 0): {np.sum(p_real_0 == 0)}")
print(f"Documentos con max_y p(y | x) = 1 exacto -> α = 0: {np.sum(P0.max(axis=1) == 1)} | α = 1: {np.sum(P1.max(axis=1) == 1)}")

In [ ]:
alpha_values = [0, 1e-4, 1e-3, 1e-2, 1e-1, 0.5, 1]

for alpha in alpha_values:
    model = myNaiveBayesBernoulli(alpha=alpha).fit(X_train_txt, y_train_txt)
    print(f"α = {alpha} | Accuracy Test: {accuracy_score(y_test_txt, model.predict(X_test_txt)):.4f}")